In [9]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer

# 1. Load the dataset
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)

# To simulate "Missing Values" for Task 2 (since the original is clean), 
# we will intentionally introduce some NaN values in the 'mean radius' column.
np.random.seed(42)
missing_indices = np.random.choice(df.index, size=20, replace=False)
df.loc[missing_indices, 'mean radius'] = np.nan

print("Dataset Shape:", df.shape)

Dataset Shape: (569, 30)


In [10]:
# --- Task 1: Identify Data Quality Issues ---
print("--- Task 1: Missing Values Check ---")
missing_vals = df.isnull().sum()
print(missing_vals[missing_vals > 0])

# --- Task 2: Apply Missing Value Strategy ---
# Strategy: Mean Imputation
# Why? Because 'mean radius' is a continuous numerical feature, and the mean is a good representative of central tendency when outliers are not extreme.

mean_value = df['mean radius'].mean()
df['mean radius'].fillna(mean_value, inplace=True)

print("\n--- Task 2: After Imputation ---")
print("Missing values in 'mean radius':", df['mean radius'].isnull().sum())

--- Task 1: Missing Values Check ---
mean radius    20
dtype: int64

--- Task 2: After Imputation ---
Missing values in 'mean radius': 0


/tmp/ipykernel_293/2234232445.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['mean radius'].fillna(mean_value, inplace=True)


## Task 3: Detect and Handle Outliers using IQR
We will calculate the Interquartile Range (IQR) for the 'mean area' feature and cap the outliers to the upper and lower bounds.

In [5]:
# Calculate Q1, Q3, and IQR
Q1 = df['mean area'].quantile(0.25)
Q3 = df['mean area'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Handle Outliers using Capping (Winsorization)
df['mean area'] = np.where(df['mean area'] > upper_bound, upper_bound, df['mean area'])
df['mean area'] = np.where(df['mean area'] < lower_bound, lower_bound, df['mean area'])

print("Outliers handled successfully using IQR bounds.")

Outliers handled successfully using IQR bounds.


## Task 4: Normalize Numerical Features
Applying both Min-Max Scaling (values between 0 and 1) and Z-score Standardization (mean=0, std=1) using scikit-learn.

In [6]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# 1. Min-Max Scaling
min_max_scaler = MinMaxScaler()
df_minmax = pd.DataFrame(min_max_scaler.fit_transform(df), columns=df.columns)

# 2. Z-score Standardization
z_scaler = StandardScaler()
df_zscore = pd.DataFrame(z_scaler.fit_transform(df), columns=df.columns)

print("Normalization completed.")
print("\n--- Min-Max Scaled Preview ---")
print(df_minmax[['mean radius', 'mean area']].head(3))

Normalization completed.

--- Min-Max Scaled Preview ---
   mean radius  mean area
0     0.504383   0.724975
1     0.630736   0.999746
2     0.587639   0.895756


## Task 5: Apply PCA
Checking correlation first. Since features like 'mean radius' and 'mean perimeter' are highly correlated, applying Principal Component Analysis (PCA) is highly effective for dimensionality reduction.

In [8]:
from sklearn.decomposition import PCA

# 1. Check Correlation
correlation_matrix = df_zscore.corr()
print("Correlation between Radius and Perimeter:", correlation_matrix.loc['mean radius', 'mean perimeter'])

# 2. Apply PCA (Reducing to 2 components for simplicity)
pca = PCA(n_components=2)
pca_result = pca.fit_transform(df_zscore)

# Create a new DataFrame for the PCA results
df_pca = pd.DataFrame(data=pca_result, columns=['PC1', 'PC2'])

print("\n--- PCA Applied Successfully ---")
print("Explained Variance Ratio:", pca.explained_variance_ratio_)
print(df_pca.head())

Correlation between Radius and Perimeter: 0.9753589518729259

--- PCA Applied Successfully ---
Explained Variance Ratio: [0.44088491 0.18913294]
        PC1        PC2
0  9.253475   1.856654
1  2.451076  -3.872642
2  5.801367  -1.184877
3  7.163069  10.233438
4  3.997386  -2.031568
